In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.model_selection import train_test_split

df = pd.read_csv("course_lead_scoring_2026.csv")
df.columns = df.columns.str.lower().str.replace(" ", "_")

target = "converted"
categorical = df.drop(columns=[target]).select_dtypes(exclude="number").columns.tolist()
numerical = [c for c in df.columns if c not in categorical + [target]]

# --- Data preparation
print("Hiányzó értékek:\n", df.isnull().sum(), "\n")
df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)

# --- Q1
print("Q1 industry módusz:", df["industry"].mode()[0])

# --- Q2
corr = df[numerical].corr()
pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
print("\nQ2 korrelációk:")
for a, b in pairs:
    print(f"  {a} ~ {b}: {corr.loc[a, b]:.4f}")

# --- Split
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train[target].values
y_val = df_val[target].values
df_train = df_train.drop(columns=[target])
df_val = df_val.drop(columns=[target])

# --- Q3
print("\nQ3 mutual information (train):")
for c in categorical:
    print(f"  {c}: {round(mutual_info_score(df_train[c], y_train), 2)}")


# --- Q4
def fit_eval(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient="records"))
    X_val = dv.transform(df_val[features].to_dict(orient="records"))
    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))


features = categorical + numerical
base_acc = fit_eval(features)
print(f"\nQ4 accuracy: {base_acc:.4f} -> {round(base_acc, 2)}")

# --- Q5
print("\nQ5 feature elimination (eltérés = eredeti - nélküle):")
for f in features:
    acc = fit_eval([x for x in features if x != f])
    print(f"  {f}: acc={acc:.4f}, diff={base_acc - acc:.4f}")

# --- Q6
print("\nQ6 regularizáció:")
results = {}
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    results[C] = round(fit_eval(features, C=C), 3)
    print(f"  C={C}: {results[C]}")
best = max(results.values())
print("Legjobb C:", min(c for c, a in results.items() if a == best))